# Root-trait ITV protocol (multi-trait, noise-aware, permutation-null)

(SRL, RD, RTD, RN) from the Han et al. (2025) compilation, plus the Weemstra gradient as external data.

**Set the data folder** (`ITV_DATA`) to the folder holding `Source_data.xlsx`, `Combined_SRL_ITV_Han_Weemstra_v2.xlsx` and `Weemstra_ITV.csv`. Cells write CSVs to the working folder.
Run time on one core is about 1.5 hours; cells 8 (leave-one-site-out and repeated CV) and 6 (permutation null and simulation) dominate.

| Step | What it does | Paper |
|---|---|---|
| 0 | Regression test: reproduces the v5 numbers (mixed model, likelihood ratio, legacy-fold elastic-net CV) | 2.8 |
| 1 | Data preparation and row-alignment checks | 2.1 |
| 2 | Variance partition, sampling noise (n-only vs plug-in), MAT/latitude mixed-model tests | 3.1, 3.3, 3.8 |
| 3 | Site-grouped nested CV, pooled skill against the training mean with a site-cluster bootstrap | 3.3, 3.8 |
| 4 | Noise-aware CV (bias-corrected CV, n-only precision weights) | 3.3 |
| 5 | External tests: Weemstra, and the authors' own forests as a held-out origin | 3.4, 3.7 |
| 6 | Site-permutation null and design simulation (two bracketing scenarios) | 3.2 |
| 7 | Weemstra within-species slope vs the Han slope, with power | 3.4, 3.7 |
| 8 | Leave-one-site-out and repeated random site partitions | 3.3 |
| 9 | GRooT overlap-threshold grid (needs the harmonized GRooT file) | 3.5, 3.6 |

**Known caveats.** The sampling-noise share of CV is not identifiable from these data (24% with variances that depend on n only, 58% with the plug-in estimate, which is biased because it uses each record's own CV);
the cells report both. `scikit-learn >= 1.6` assigns tied groups to GroupKFold folds differently; `LegacyGroupKFold` pins the historical assignment.

In [ ]:
import os, sys, subprocess
os.environ.setdefault('ITV_DATA', './data')   # <- folder with the three input files
sys.path.insert(0, '.')
import itvcheck; print('itvcheck loaded from', itvcheck.__file__)

## 0. Regression test against the published v5 numbers

In [ ]:
exec(open('tests/test_validate.py').read().replace('os.path.dirname(__file__)', "'.'"))

## 1. Data preparation

In [ ]:
import os, numpy as np, pandas as pd
U=os.environ.get('ITV_DATA','./data').rstrip('/')+'/'
g=pd.read_excel(U+'Source_data.xlsx','root ITV globe'); c=pd.read_excel(U+'Combined_SRL_ITV_Han_Weemstra_v2.xlsx')
ch=c[c.data_source=='Han_2025'].reset_index(drop=True)
assert len(g)==len(ch)==543 and np.allclose(g['cv.SRL'].fillna(-1),ch.CV_SRL.fillna(-1),atol=1e-4) and (g.MAT.round(1)==ch.MAT.round(1)).all()   # row alignment of the two files
d=pd.DataFrame({'species':ch.species.str.replace(r'_\d+$','',regex=True).str.replace(' ','_'),'site':ch.site_cluster,'origin':g.data_origin,'study':g.study_ID,'MAT':g.MAT,'MAP':g.MAP,'abs_lat':g['abl.lat'],'woody':g.plant_woodiness,'symb':g.symbiotic_status,'family':g.family,'n':g.n,
  **{f'cv_{t}':g[f'cv.{t}'] for t in ['SRL','RD','RTD','RN']}, **{f'mean_{t}':g[t] for t in ['SRL','RD','RTD','RN']}})
d.to_pickle('han.pkl'); print(d.shape)

## 2. Variance partition, sampling noise and MAT / latitude tests

In [ ]:
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, time
from itvcheck import *
from scipy import optimize
d=pd.read_pickle('han.pkl'); TR=['SRL','RD','RTD','RN']; t0=time.time()
# ---- F1: outcome-independent noise model (n only) -> variance partition and MAT tests
rows=[]; tests=[]
for t in TR:
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); y=s[f'cv_{t}'].values; n=s.n.values; V=np.var(y,ddof=1)
    ystar=bias_correct_cv(y,n); vn=cv_noise_var_n(ystar.mean(),n); plug=cv_sampling_var(y,n); sg=[s.site.values,s.species.values]; X0=np.ones((len(s),1))
    zM=((s.MAT-s.MAT.mean())/s.MAT.std()).values; zL=((s.abs_lat-s.abs_lat.mean())/s.abs_lat.std()).values
    a=fit_reml(y,X0,sg); b=fit_reml(ystar,X0,sg,meas_var=vn); tot=b.vcomp.sum()+b.scale
    row={'trait':t,'noise_share_n_only':vn.mean()/np.var(ystar,ddof=1),'noise_share_plugin':plug.mean()/V,'ceiling_R2_range_lo':1-plug.mean()/V,'ceiling_R2_range_hi':1-vn.mean()/np.var(ystar,ddof=1),
         'plain_site':a.vcomp[0]/(a.vcomp.sum()+a.scale),'plain_species':a.vcomp[1]/(a.vcomp.sum()+a.scale),'plain_resid':a.scale/(a.vcomp.sum()+a.scale),
         'true_site_of_true':b.vcomp[0]/tot,'true_species_of_true':b.vcomp[1]/tot,'true_resid_of_true':b.scale/tot}
    for tag,yy,mv in [('plain',y,None),('noise_n_only',ystar,vn)]:
        fits={k:fit_reml(yy,(np.c_[np.ones(len(s)),*c] if c else X0),sg,meas_var=mv,reml=False) for k,c in {'null':[],'MAT':[zM],'lat':[zL],'MAT+lat':[zM,zL]}.items()}
        for a_,b_ in [('null','MAT'),('null','lat'),('MAT','MAT+lat'),('lat','MAT+lat')]:
            lr,df,p=lr_test(fits[a_],fits[b_]); tests.append({'trait':t,'model':tag,'test':f'{a_}->{b_}','LR':lr,'p':p})
        for k in fits: tests.append({'trait':t,'model':tag,'test':f'AIC_{k}','LR':-2*fits[k].llf+2*(fits[k].p+3),'p':np.nan})
        rM=fit_reml(yy,np.c_[np.ones(len(s)),zM],sg,meas_var=mv); fx=np.var(np.c_[np.ones(len(s)),zM]@rM.beta); row[f'MAT_beta_per_SD_{tag}']=rM.beta[1]; row[f'MAT_t_{tag}']=rM.t[1]
        row[f'MAT_share_of_true_var_{tag}']=fx/(fx+rM.vcomp.sum()+rM.scale)
    rows.append(row)
pd.DataFrame(rows).to_csv('F1_noise_n_only_partition.csv',index=False); pd.DataFrame(tests).to_csv('F1_lmm_tests.csv',index=False); print('F1 done',round(time.time()-t0),flush=True)

## 3. Site-grouped nested CV and pooled skill, four traits

In [ ]:
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd
from itvcheck import *
d=pd.read_pickle('han.pkl'); rows=[]; folds_rows=[]
for t in ['SRL','RD','RTD','RN']:
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); X=s[NUMERIC+LOWCARD].copy(); g=s.site.values; n=s.n.values
    sp=list(LegacyGroupKFold(5).split(X,s[f'cv_{t}'],groups=g))
    for variant,y,wfit in [('raw',s[f'cv_{t}'].values,None)]:
        o=grouped_oof(X,y,g,sp,w=wfit); 
        # evaluate with precision weights in BOTH variants (weights are always defined from the sampling-variance model)
        sk=skill_table(o,B=1000); sk.insert(0,'variant',variant); sk.insert(0,'trait',t); rows.append(sk)
        for (m,f),q in o.groupby(['model','fold']): folds_rows.append({'trait':t,'variant':variant,'model':m,'fold':f,'n':len(q),'r2':r2_score(q.y,q.pred),'skill':1-((q.y-q.pred)**2).sum()/((q.y-q.train_mean)**2).sum()})
        print(t,variant,'done',flush=True)
R=pd.concat(rows); R.to_csv('A_cv_by_trait.csv',index=False); pd.DataFrame(folds_rows).to_csv('A_cv_folds_by_trait.csv',index=False)
pd.set_option('display.width',250); print(R.round(3).to_string(index=False))

## 4. Noise-aware CV (bias-corrected CV, n-only precision weights)

In [ ]:
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, time
from itvcheck import *
from scipy import optimize
d=pd.read_pickle('han.pkl'); TR=['SRL','RD','RTD','RN']; t0=time.time()
# ---- F2: noise-aware CV with n-only precision weights (outcome-independent)
res=[]
for t in TR:
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); X=s[NUMERIC+LOWCARD].copy(); g=s.site.values; n=s.n.values; y=s[f'cv_{t}'].values
    sp=list(LegacyGroupKFold(5).split(X,y,groups=g)); ystar=bias_correct_cv(y,n); vn=cv_noise_var_n(ystar.mean(),n); tau2=max(np.var(ystar,ddof=1)-vn.mean(),0.15*np.var(ystar,ddof=1)); w=1/(tau2+vn); w=w/w.mean()
    o=grouped_oof(X,ystar,g,sp,w=w,n_jobs=1); sk=skill_table(o,w=w,B=1000); sk.insert(0,'variant','bias-corrected CV + n-only precision weights'); sk.insert(0,'trait',t); res.append(sk); print('F2',t,round(time.time()-t0),flush=True)
pd.concat(res).to_csv('F2_noise_aware_cv_n_only.csv',index=False)

## 5. External tests (Weemstra; the authors' own forests as a held-out origin)

In [ ]:
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd
from itvcheck import *
U=os.environ.get('ITV_DATA','./data').rstrip('/')+'/'
d=pd.read_pickle('han.pkl'); B=None
c=pd.read_excel(U+'Combined_SRL_ITV_Han_Weemstra_v2.xlsx'); wm=c[c.data_source=='Weemstra_2021'][['species','altitude_m','MAT','MAP','abl.lat','plant_woodiness','symbiotic_status']].rename(columns={'abl.lat':'abs_lat','plant_woodiness':'woody','symbiotic_status':'symb'})
wi=pd.read_csv(U+'Weemstra_ITV.csv'); W=wi.merge(wm,on=['species','altitude_m'],how='left'); assert W.MAT.notna().all() and len(W)==88, (W.MAT.isna().sum(),len(W))
print('Weemstra MAT range',W.MAT.min(),W.MAT.max(),'altitudes',W.altitude_m.nunique(),'distinct MAT',W.MAT.nunique(),'species',W.species.nunique())
TR={'SRL':[('corrected','CV_SRL')],'RD':[('corrected','CV_RD'),('direct','CV_RD_direct')],'RTD':[('corrected','CV_RTD'),('direct','CV_RTD_direct')]}
rows=[]; slope_rows=[]; r=np.random.default_rng(42)
def fit_models(s,col,noise_aware):
    X=s[NUMERIC+LOWCARD].copy(); y=s[col].values; g=s.site.values; n=s.n.values; w=None
    if noise_aware:
        pass
    out={}
    for m,(est,grid) in make_models().items():
        pipe=Pipeline([('prep',prep()),('model',est)]); inner=list(LegacyGroupKFold(5).split(X,y,groups=g)); gs=GridSearchCV(pipe,grid,cv=inner,scoring='r2',n_jobs=1)
        gs.fit(X,y) if w is None else gs.fit(X,y,model__sample_weight=w); out[m]=gs.best_estimator_
    return out,y,w
for t,cols in TR.items():
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); mu,sd=s.MAT.mean(),s.MAT.std(); zs=(s.MAT-mu)/sd; sg=[s.site.values,s.species.values]
    for variant in ['raw']:
        mods,ytr,wtr=fit_models(s,f'cv_{t}',False)
        mv=None
        l0=fit_reml(ytr,np.ones((len(s),1)),sg,meas_var=mv); l1=fit_reml(ytr,np.c_[np.ones(len(s)),zs],sg,meas_var=mv)
        trmean=np.average(ytr,weights=wtr)
        for calc,col in cols:
            y=W[col].values; preds={m:mods[m].predict(W[NUMERIC+LOWCARD]) for m in mods}; preds['TrainMean']=np.full(len(W),trmean); preds['LMM_intercept']=np.full(len(W),l0.beta[0]); preds['LMM_MAT']=l1.beta[0]+l1.beta[1]*((W.MAT-mu)/sd).values
            wW=np.ones(len(y)); 
            mse={k:((y-p)**2) for k,p in preds.items()}; clim=W.altitude_m.values; ualt=np.unique(clim)
            def boot(a,b,wt=None,B=2000):
                rr=np.random.default_rng(42); out=[]
                for _ in range(B):
                    ix=np.concatenate([np.where(clim==u)[0] for u in rr.choice(ualt,len(ualt),replace=True)]); ww=np.ones(len(ix)) if wt is None else wt[ix]; out.append(np.average(a[ix],weights=ww)-np.average(b[ix],weights=ww))
                return np.percentile(out,[2.5,97.5])
            for k in preds:
                row={'trait':t,'calc':calc,'train':variant,'model':k,'rmse':np.sqrt(mse[k].mean()),'rmse_w':np.sqrt(np.average(mse[k],weights=wW)),'bias_obs_minus_pred':(y-preds[k]).mean()}
                for ref in ['TrainMean','LMM_intercept']:
                    if k!=ref: lo,hi=boot(mse[k],mse[ref]); row[f'dMSE_vs_{ref}']=mse[k].mean()-mse[ref].mean(); row[f'dMSE_vs_{ref}_lo']=lo; row[f'dMSE_vs_{ref}_hi']=hi
                rows.append(row)
    print(t,'external models done',flush=True)
pd.DataFrame(rows).to_csv('E_weemstra_external.csv',index=False)
# ---- Han's own in-situ campaigns (Ours_9/10/11) as a held-out origin: train on Global rows only
orows=[]
for t in ['SRL','RD','RTD','RN']:
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); tr=s[s.origin=='Global'].reset_index(drop=True); te=s[s.origin=='Ours'].reset_index(drop=True)
    X=tr[NUMERIC+LOWCARD]; y=tr[f'cv_{t}'].values; g=tr.site.values
    for m,(est,grid) in make_models().items():
        pipe=Pipeline([('prep',prep()),('model',est)]); gs=GridSearchCV(pipe,grid,cv=list(LegacyGroupKFold(5).split(X,y,groups=g)),scoring='r2',n_jobs=1).fit(X,y); p=gs.predict(te[NUMERIC+LOWCARD])
        for site,q in te.assign(p=p).groupby('site'):
            orows.append({'trait':t,'model':m,'site':site,'n':len(q),'mean_obs':q[f'cv_{t}'].mean(),'mean_pred':q.p.mean(),'sse':((q[f'cv_{t}']-q.p)**2).sum(),'sse_trainmean':((q[f'cv_{t}']-y.mean())**2).sum()})
O=pd.DataFrame(orows); O.to_csv('E_ours_holdout_by_site.csv',index=False)
T=O.groupby(['trait','model']).agg(n=('n','sum'),sse=('sse','sum'),sse0=('sse_trainmean','sum')); T['skill']=1-T.sse/T.sse0; T['rmse']=np.sqrt(T.sse/T.n); T['rmse_trainmean']=np.sqrt(T.sse0/T.n); T.to_csv('E_ours_holdout_summary.csv')
pd.set_option('display.width',250); pd.set_option('display.max_columns',40)
print(T.round(3).to_string())
E=pd.DataFrame(rows); print(E[E.train=='raw'].round(4).to_string(index=False))

## 6. Site-permutation null and design simulation  *(set NP, NS as arguments: defaults 300 and 100)*

In [ ]:
sys.argv = ['06', '300', '100']
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, time
from itvcheck import *
from scipy import optimize
d=pd.read_pickle('han.pkl'); TR=['SRL','RD','RTD','RN']; t0=time.time()
# ---- F3: site-level permutation null for the elastic-net CV statistics (assumption-light)
rows=[]; NP=int(sys.argv[1]) if len(sys.argv)>1 else 300
for t in TR:
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); y=s[f'cv_{t}'].values; g=s.site.values; NUM=s[NUMERIC].values.astype(float); OH=OneHotEncoder(handle_unknown='ignore').fit(s[LOWCARD]).transform(s[LOWCARD]).toarray()
    folds=list(LegacyGroupKFold(5).split(NUM,y,groups=g)); inner=fold_cache(g,folds); obs=fast_en_cv(NUM,OH,y,folds,inner)
    sites=np.unique(g); trip={k:NUM[g==k][0] for k in sites}; r=np.random.default_rng(2024); null=[]
    for _ in range(NP):
        perm=r.permutation(sites); mp=dict(zip(sites,perm)); Np=np.array([trip[mp[k]] for k in g]); null.append(fast_en_cv(Np,OH,y,folds,inner))
    N=np.array(null); rows.append({'trait':t,'n_perm':NP,'obs_meanfoldR2':obs[0],'obs_skill':obs[1],'obs_beats':obs[2],'obs_pooledR2':obs[3],
        'null_meanfoldR2_mean':N[:,0].mean(),'null_meanfoldR2_p2_5':np.percentile(N[:,0],2.5),'null_meanfoldR2_p97_5':np.percentile(N[:,0],97.5),'p_R2_ge_obs':(1+(N[:,0]>=obs[0]).sum())/(NP+1),
        'null_skill_mean':N[:,1].mean(),'null_skill_p2_5':np.percentile(N[:,1],2.5),'null_skill_p97_5':np.percentile(N[:,1],97.5),'p_skill_ge_obs':(1+(N[:,1]>=obs[1]).sum())/(NP+1),'p_beats_ge_obs':(1+(N[:,2]>=obs[2]).sum())/(NP+1)})
    print('F3',t,rows[-1],round(time.time()-t0),flush=True)
pd.DataFrame(rows).to_csv('F3_permutation_null.csv',index=False)
# ---- F4: design simulation, two bracketing decompositions of the non-climate variance
s=d[d.cv_SRL.notna()].reset_index(drop=True); y0=s.cv_SRL.values; n=s.n.values; V=np.var(y0,ddof=1); g=s.site.values; spp=s.species.values
NUM=s[NUMERIC].values.astype(float); OH=OneHotEncoder(handle_unknown='ignore').fit(s[LOWCARD]).transform(s[LOWCARD]).toarray(); folds=list(LegacyGroupKFold(5).split(NUM,y0,groups=g)); inner=fold_cache(g,folds)
plain=fit_reml(y0,np.ones((len(s),1)),[g,spp]); ys=bias_correct_cv(y0,n); vn=cv_noise_var_n(ys.mean(),n); nz=fit_reml(ys,np.ones((len(s),1)),[g,spp],meas_var=vn)
SC={'S1_plain (all non-climate variance is real: site 49%, species 20%, residual 32%)':dict(site=plain.vcomp[0],sp=plain.vcomp[1],res=plain.scale,noise=np.zeros(len(s))),
    'S2_noise (n-only sampling noise 24%, rest real)':dict(site=nz.vcomp[0],sp=nz.vcomp[1],res=nz.scale,noise=vn)}
zrow=((s.MAT-s.MAT.mean())/s.MAT.std()).values; out=[]; NS=int(sys.argv[2]) if len(sys.argv)>2 else 100
for name,P in SC.items():
    for share in [0,0.05,0.10,0.15,0.20,0.30,0.40]:
        r=np.random.default_rng(int(1000*share)+5); res=[]
        for k in range(NS):
            su=dict(zip(np.unique(g),r.normal(0,1,len(np.unique(g))))); spu=dict(zip(np.unique(spp),r.normal(0,1,len(np.unique(spp)))))
            clim=np.sqrt(share*V)*zrow; idio=max(P['site']-share*V,0)
            y=y0.mean()+clim+np.sqrt(idio)*np.array([su[x] for x in g])+np.sqrt(P['sp'])*np.array([spu[x] for x in spp])+r.normal(0,np.sqrt(P['res']),len(y0))+r.normal(0,1,len(y0))*np.sqrt(P['noise'])
            res.append(fast_en_cv(NUM,OH,y,folds,inner)+(float(np.var(clim)/np.var(y,ddof=1)),))
        a=np.array(res); out.append({'scenario':name,'true_share_of_total_var':share,'realized_share':a[:,4].mean(),'exp_mean_fold_R2':a[:,0].mean(),'P_meanR2_gt0':(a[:,0]>0).mean(),'exp_pooled_skill':a[:,1].mean(),'P_skill_gt0':(a[:,1]>0).mean(),'P_beat_ge4of5':(a[:,2]>=4).mean(),'n_sims':NS})
    print('F4',name,round(time.time()-t0),flush=True)
pd.DataFrame(out).to_csv('F4_design_simulation_two_scenarios.csv',index=False); print('ALL DONE')

## 7. Weemstra within-species slope vs the Han slope, with power

In [ ]:
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd
from itvcheck import *
U=os.environ.get('ITV_DATA','./data').rstrip('/')+'/'
d=pd.read_pickle('han.pkl'); c=pd.read_excel(U+'Combined_SRL_ITV_Han_Weemstra_v2.xlsx'); wm=c[c.data_source=='Weemstra_2021'][['species','altitude_m','MAT']]
W=pd.read_csv(U+'Weemstra_ITV.csv').merge(wm,on=['species','altitude_m']); rows=[]
def slope(y,x,sp,clim,B=2000):
    df=pd.DataFrame({'y':y,'x':x,'sp':sp}); yc=df.y-df.groupby('sp').y.transform('mean'); xc=df.x-df.groupby('sp').x.transform('mean'); b=(xc*yc).sum()/(xc**2).sum(); res=yc-b*xc
    se=np.sqrt((res**2).sum()/(len(df)-df.sp.nunique()-1)/(xc**2).sum()); ua=np.unique(clim); rr=np.random.default_rng(42); bs=[]
    for _ in range(B):
        ix=np.concatenate([np.where(clim==u)[0] for u in rr.choice(ua,len(ua),replace=True)]); dd=df.iloc[ix]; a=dd.y-dd.groupby('sp').y.transform('mean'); q=dd.x-dd.groupby('sp').x.transform('mean'); bs.append((q*a).sum()/(q**2).sum() if (q**2).sum()>0 else np.nan)
    return b,se,np.nanpercentile(bs,[2.5,97.5]),res.std(ddof=1),(xc**2).sum()
for t,cols in {'SRL':[('corrected','CV_SRL')],'RD':[('corrected','CV_RD'),('direct','CV_RD_direct')],'RTD':[('corrected','CV_RTD'),('direct','CV_RTD_direct')]}.items():
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); y=s[f'cv_{t}'].values; sg=[s.site.values,s.species.values]; zs=(s.MAT-s.MAT.mean())/s.MAT.std()
    f=fit_reml(y,np.c_[np.ones(len(s)),zs],sg); hb=f.beta[1]/s.MAT.std(); hse=f.se[1]/s.MAT.std()    # plain mixed model (noise model does not change the slope: see F1)
    for calc,col in cols:
        b,se,ci,rs,sxx=slope(W[col].values,W.MAT.values,W.species.values,W.altitude_m.values); dfree=len(W)-W.species.nunique()-1; tc=stats.t.ppf(.975,dfree)
        pw=lambda beta:float(1-stats.nct.cdf(tc,dfree,beta/np.sqrt(rs**2/sxx))+stats.nct.cdf(-tc,dfree,beta/np.sqrt(rs**2/sxx)))
        rows.append({'trait':t,'calc':calc,'weemstra_slope_per_C':b,'ci_lo':ci[0],'ci_hi':ci[1],'han_slope_per_C':hb,'han_lo':hb-1.96*hse,'han_hi':hb+1.96*hse,'gradient_C':W.MAT.max()-W.MAT.min(),
                     'power_han_slope':pw(hb),'power_han_lower95':pw(max(hb-1.96*hse,0)),'min_detectable_slope_80':float(2.8*np.sqrt(rs**2/sxx)),'z_diff':(b-hb)/np.sqrt(se**2+hse**2)})
pd.DataFrame(rows).to_csv('G_weemstra_slopes_plainLMM.csv',index=False); pd.set_option('display.width',250); print(pd.DataFrame(rows).round(4).T.to_string())

## 8. Leave-one-site-out and repeated random site partitions

In [ ]:
import os, sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd
from itvcheck import *
d=pd.read_pickle('han.pkl'); rows=[]; rep_rows=[]; R=20
def rand_folds(g,rep,k=5):
    r=np.random.default_rng(1000+rep); ch=np.array_split(r.permutation(np.unique(g)),k); return [(np.where(~np.isin(g,c))[0],np.where(np.isin(g,c))[0]) for c in ch]
for t in ['SRL','RD','RTD','RN']:
    s=d[d[f'cv_{t}'].notna()].reset_index(drop=True); X=s[NUMERIC+LOWCARD].copy(); g=s.site.values; y=s[f'cv_{t}'].values
    en={'ElasticNet':make_models()['ElasticNet']}
    o=grouped_oof(X,y,g,list(LeaveOneGroupOut().split(X,y,groups=g)),models=en,n_jobs=1); sk=skill_table(o,B=1000); sk.insert(0,'design','LOSO'); sk.insert(0,'trait',t); rows.append(sk)
    for rep in range(R):
        o=grouped_oof(X,y,g,rand_folds(g,rep),models=en,n_jobs=1); q=o.groupby('fold').apply(lambda z: r2_score(z.y,z.pred)); 
        rep_rows.append({'trait':t,'rep':rep,'model':'ElasticNet','mean_fold_R2':q.mean(),'pooled_skill':1-((o.y-o.pred)**2).sum()/((o.y-o.train_mean)**2).sum(),'pooled_R2':r2_score(o.y,o.pred)})
    print(t,'EN LOSO + repeated done',flush=True)
# random forest on SRL only (150 trees): LOSO + 8 repeats
s=d[d.cv_SRL.notna()].reset_index(drop=True); X=s[NUMERIC+LOWCARD].copy(); g=s.site.values; y=s.cv_SRL.values
rf={'RandomForest':make_models(n_trees=150)['RandomForest']}
o=grouped_oof(X,y,g,list(LeaveOneGroupOut().split(X,y,groups=g)),models=rf,n_jobs=1); sk=skill_table(o,B=1000); sk.insert(0,'design','LOSO'); sk.insert(0,'trait','SRL'); rows.append(sk); print('RF LOSO done',flush=True)
for rep in range(8):
    o=grouped_oof(X,y,g,rand_folds(g,rep),models=rf,n_jobs=1); q=o.groupby('fold').apply(lambda z: r2_score(z.y,z.pred))
    rep_rows.append({'trait':'SRL','rep':rep,'model':'RandomForest','mean_fold_R2':q.mean(),'pooled_skill':1-((o.y-o.pred)**2).sum()/((o.y-o.train_mean)**2).sum(),'pooled_R2':r2_score(o.y,o.pred)})
pd.concat(rows).to_csv('C_loso_by_trait.csv',index=False); rp=pd.DataFrame(rep_rows); rp.to_csv('C_repeated_cv.csv',index=False)
pd.set_option('display.width',250); print(pd.concat(rows).round(3).to_string(index=False))
print(rp.groupby(['trait','model']).agg(R2_mean=('mean_fold_R2','mean'),R2_lo=('mean_fold_R2',lambda s:s.quantile(.025)),R2_hi=('mean_fold_R2',lambda s:s.quantile(.975)),frac_R2_pos=('mean_fold_R2',lambda s:(s>0).mean()),skill_mean=('pooled_skill','mean'),skill_lo=('pooled_skill',lambda s:s.quantile(.025)),skill_hi=('pooled_skill',lambda s:s.quantile(.975)),frac_skill_pos=('pooled_skill',lambda s:(s>0).mean())).round(3).to_string())

## 9. GRooT overlap-threshold grid (guarded: skipped when the harmonized GRooT file is absent)

In [ ]:
import pandas as pd
from groot_overlap_grid import groot_overlap_grid
grid = groot_overlap_grid(os.path.join(os.environ['ITV_DATA'], 'GRooT_ITV_harmonized.csv'), pd.read_pickle('han.pkl'))
if grid is not None: print(grid.to_string(index=False))